[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/S09_04_q_learning.ipynb)

# S09 · 04 — Aprendizaje por refuerzo: Q-learning en el hotel

**Tiempo estimado:** 1 h · **Unidad del temario:** 3.12 (introducción al aprendizaje por refuerzo)

**Contexto TurisData.** En el aprendizaje supervisado alguien nos da la respuesta correcta. Aquí **no**:
un **agente** prueba acciones, recibe **recompensas** (o castigos) y aprende por ensayo y error una
**política** (qué hacer en cada situación). Vamos a entrenar a un "botones virtual" que debe llevar a
un huésped desde recepción hasta la habitación con vistas al mar, evitando la zona en obras.
Es un juguete, pero contiene las piezas de cualquier problema de refuerzo (robots, juegos, precios dinámicos).

**Al terminar sabrás:**
1. Los conceptos: estado, acción, recompensa, política, episodio.
2. La regla de actualización de **Q-learning** y qué significan `alpha` y `gamma`.
3. El equilibrio **exploración / explotación** (epsilon-greedy).
4. Leer la tabla Q como una política y comprobar si el agente ha aprendido.

*Todo está en numpy. En proyectos reales se usa la librería Gymnasium, que ofrece la misma interfaz
(`reset` / `step`) que el entorno que escribimos aquí.*

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="white")

## 1. El entorno: el pasillo del hotel

Una cuadrícula de 4 filas × 5 columnas:

```
  S  .  .  .  .        S = recepción (inicio)
  .  #  #  .  X        # = pared (no se puede cruzar)
  .  .  .  .  .        X = zona en obras (recompensa -5, pero se puede pisar)
  .  #  .  .  G        G = habitación con vistas al mar (recompensa +10, fin del episodio)
```

- **Estado:** la casilla donde está el agente (20 posibles, numeradas de 0 a 19 por filas).
- **Acciones:** 0 = arriba, 1 = abajo, 2 = izquierda, 3 = derecha.
- **Recompensa:** -1 por cada paso (el huésped tiene prisa), -5 extra en obras, +10 al llegar a G.
- Si el agente choca con una pared o el borde, **se queda donde está** (y paga el paso).
- **Episodio:** una partida, desde recepción hasta llegar a G (o hasta 50 pasos).

In [ ]:
class EntornoHotel:
    """Cuadrícula sencilla con la misma interfaz que Gymnasium: reset() y step(accion)."""
    FILAS, COLUMNAS = 4, 5
    N_ESTADOS, N_ACCIONES = 20, 4
    PAREDES = {(1, 1), (1, 2), (3, 1)}
    OBRAS = (1, 4)
    META = (3, 4)
    MOVIMIENTOS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
    NOMBRES_ACCION = ["arriba", "abajo", "izquierda", "derecha"]

    def __init__(self, coste_paso=1.0):
        self.coste_paso = coste_paso      # lo que "cuesta" cada paso (1.0 por defecto)

    def reset(self):
        self.pos = (0, 0)
        return self.estado()

    def estado(self):
        return self.pos[0] * self.COLUMNAS + self.pos[1]

    def step(self, accion):
        df, dc = self.MOVIMIENTOS[accion]
        nueva = (self.pos[0] + df, self.pos[1] + dc)
        dentro = 0 <= nueva[0] < self.FILAS and 0 <= nueva[1] < self.COLUMNAS
        if dentro and nueva not in self.PAREDES:
            self.pos = nueva
        recompensa = -self.coste_paso
        if self.pos == self.OBRAS:
            recompensa -= 5.0
        terminado = self.pos == self.META
        if terminado:
            recompensa += 10.0
        return self.estado(), recompensa, terminado


entorno = EntornoHotel()
s = entorno.reset()
print("Estado inicial:", s)
for a in (3, 3, 1):   # derecha, derecha, abajo (choca con la pared)
    s, r, fin = entorno.step(a)
    print(f"acción {EntornoHotel.NOMBRES_ACCION[a]:>9} → estado {s:>2}, recompensa {r:+.0f}, fin={fin}")

## 2. La tabla Q y su regla de actualización

**Idea.** Guardamos una tabla `Q[estado, acción]` con *"cuánta recompensa total espero si estoy aquí y
hago esta acción y luego sigo lo mejor posible"*. Empieza en ceros. Tras cada paso se corrige un poco:

`Q[s,a] ← Q[s,a] + alpha · ( r + gamma · max(Q[s',:]) − Q[s,a] )`

- `alpha` (0–1): **velocidad de aprendizaje**. Alto = se cree lo último; bajo = cambia despacio.
- `gamma` (0–1): cuánto **importa el futuro**. Con 0 el agente solo mira la recompensa inmediata.
- `r + gamma·max(Q[s'])` es lo que **ahora** creemos que vale esa acción; se compara con lo que creíamos antes.

### Ejercicio 1
Escribe `actualizar_q(Q, s, a, r, s2, alpha, gamma)` que aplique la regla **modificando `Q` en su sitio**
(no hace falta devolver nada).

In [ ]:
def actualizar_q(Q, s, a, r, s2, alpha, gamma):
    # TODO: Primero calcula lo que ahora crees que vale la acción (recompensa + futuro descontado); luego mueve Q[s, a] una fracción alpha hacia ese valor
    ...

Qprueba = np.zeros((20, 4))
Qprueba[5, 3] = 4.0
actualizar_q(Qprueba, s=0, a=3, r=-1.0, s2=5, alpha=0.5, gamma=0.9)
assert abs(Qprueba[0, 3] - 1.3) < 1e-9, "Con Q[0,3]=0, r=-1, max Q[5]=4, alpha=0.5, gamma=0.9 debe salir 0.5·(-1+0.9·4)=1.3"
assert np.count_nonzero(Qprueba) == 2, "Solo debes modificar Q[s, a]"
print("Ejercicio 1 correcto ✔")

## 3. Explorar o explotar: epsilon-greedy

**Idea.** Si el agente siempre hace lo que cree mejor, nunca descubre caminos mejores. Si siempre
experimenta, nunca aprovecha lo aprendido. **Epsilon-greedy**: con probabilidad `epsilon` hace una acción
**al azar** (explorar) y con probabilidad `1 - epsilon` la **mejor conocida** (explotar).

### Ejercicio 2
Escribe `elegir_accion(Q, s, epsilon, rng)`: con probabilidad `epsilon` devuelve una acción aleatoria
(`rng.integers(4)`), y si no, la de mayor valor en `Q[s]` (`np.argmax`). Usa `rng.random()` para decidir.

In [ ]:
def elegir_accion(Q, s, epsilon, rng):
    # TODO: Un único if: si el número aleatorio es menor que epsilon, explora; en otro caso, elige la mejor acción de la fila s
    ...

rng_t = np.random.default_rng(0)
Qt = np.zeros((20, 4)); Qt[7] = [0, 0, 5, 0]
assert all(elegir_accion(Qt, 7, 0.0, rng_t) == 2 for _ in range(50)), "Con epsilon=0 debe elegir siempre la mejor acción (2)"
distintas = {elegir_accion(Qt, 7, 1.0, rng_t) for _ in range(200)}
assert distintas == {0, 1, 2, 3}, "Con epsilon=1 debe explorar las 4 acciones al azar"
print("Ejercicio 2 correcto ✔")

## 4. Entrenar al agente

Un bucle: en cada episodio, el agente empieza en recepción, elige acciones, recibe recompensas y
actualiza `Q` con **tus** dos funciones. Guardamos la recompensa total de cada episodio para ver si mejora.

In [ ]:
def entrenar(episodios=1500, alpha=0.2, gamma=0.95, epsilon=0.2, semilla=SEMILLA, max_pasos=50, coste_paso=1.0):
    rng = np.random.default_rng(semilla)
    env = EntornoHotel(coste_paso)
    Q = np.zeros((env.N_ESTADOS, env.N_ACCIONES))
    recompensas = []
    for _ in range(episodios):
        s = env.reset()
        total = 0.0
        for _ in range(max_pasos):
            a = elegir_accion(Q, s, epsilon, rng)
            s2, r, fin = env.step(a)
            actualizar_q(Q, s, a, r, s2, alpha, gamma)
            total += r
            s = s2
            if fin:
                break
        recompensas.append(total)
    return Q, np.array(recompensas)

Q, recompensas = entrenar()
print("Recompensa media de los primeros 50 episodios:", recompensas[:50].mean().round(1))
print("Recompensa media de los últimos 50 episodios: ", recompensas[-50:].mean().round(1))

In [ ]:
suave = pd.Series(recompensas).rolling(30).mean()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(recompensas, alpha=0.25, label="cada episodio")
ax.plot(suave, color="tab:red", label="media móvil (30)")
ax.set_xlabel("Episodio"); ax.set_ylabel("Recompensa total")
ax.set_title("El agente mejora con la experiencia"); ax.legend()
sns.despine(); plt.show()

## 5. Leer la política aprendida

La **política** es la acción de mayor valor en cada estado: `argmax` de cada fila de `Q`.

### Ejercicio 3
Guarda en `politica` un vector de 20 acciones (la mejor en cada estado). Después escribe el `camino`
que sigue el agente **sin explorar** desde recepción: lista de estados visitados, empezando en 0, hasta
llegar a la meta (estado 19) o hasta un máximo de 30 pasos. Usa `EntornoHotel` y `politica`.

In [ ]:
# TODO: La mejor acción de cada fila de Q (argmax por filas); para el camino, repite step con la acción que dicta la política hasta llegar a la meta
...

assert len(politica) == 20, "politica debe tener una acción por estado (20)"
assert camino[0] == 0 and camino[-1] == 19, "El camino debe empezar en 0 y llegar a la habitación (estado 19)"
assert 9 not in camino, "El camino no debería pasar por la zona en obras (estado 9)"
assert len(camino) <= 10, "El camino aprendido debería ser corto (7-9 pasos)"
print(f"Ejercicio 3 correcto ✔ — camino de {len(camino) - 1} pasos:", camino)

In [ ]:
# Dibujo de la política: flechas sobre la cuadrícula
flechas = {0: "↑", 1: "↓", 2: "←", 3: "→"}
mapa = np.full((4, 5), "", dtype=object)
fig, ax = plt.subplots(figsize=(6, 4.5))
valores = Q.max(axis=1).reshape(4, 5)
sns.heatmap(valores, annot=False, cmap="YlGn", cbar_kws={"label": "valor del estado (max Q)"}, ax=ax, linewidths=1, linecolor="white")
for f in range(4):
    for c in range(5):
        if (f, c) in EntornoHotel.PAREDES:
            texto = "PARED"
        elif (f, c) == EntornoHotel.META:
            texto = "META"
        else:
            texto = flechas[politica[f * 5 + c]] + ("\nOBRAS" if (f, c) == EntornoHotel.OBRAS else "")
        ax.text(c + 0.5, f + 0.5, texto, ha="center", va="center", fontsize=13)
ax.set_title("Política aprendida y valor de cada casilla"); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 6. Experimento: ¿qué pasa sin exploración?

Con el coste de -1 por paso, incluso el agente que nunca explora acaba aprendiendo (cada choque le
resulta "caro" y le empuja a probar otra dirección). Vamos a quitar esa ayuda: **pasos gratis**
(`coste_paso=0`). Ahora la única recompensa llega al final, en la habitación, y hay que *encontrarla*.

### Ejercicio 4
Entrena dos agentes con `entrenar(..., coste_paso=0.0)`: uno con `epsilon=0.0` (nunca explora) y otro con
`epsilon=0.3`. Guarda sus recompensas por episodio en `rec_eps0` y `rec_eps03`. Calcula `media_final_eps0`
y `media_final_eps03` como la media de sus **últimos 100 episodios** y guarda en `explorar_ayuda`
(`True`/`False`) si el agente que explora consigue mayor recompensa media final que el que no.

In [ ]:
# TODO: entrenar devuelve dos cosas (Q, recompensas): guarda la segunda; la media de los últimos 100 son las últimas posiciones del vector
...

assert len(rec_eps0) == 1500 and len(rec_eps03) == 1500, "Cada entrenamiento tiene 1500 episodios"
assert explorar_ayuda is True, "Con pasos gratis, el agente que no explora no debería llegar nunca a la meta: revisa coste_paso=0.0 y epsilon"
print(f"Sin explorar: {media_final_eps0:.1f} · Explorando (0,3): {media_final_eps03:.1f}")
_, rec_gamma0 = entrenar(gamma=0.0)
print("Con gamma=0 (sin mirar al futuro), recompensa final:", rec_gamma0[-100:].mean().round(1), "(compárala con el +3 óptimo)")

## Mini-reto integrador: cambiar las reglas del negocio

El hotel decide que **pasar por la zona en obras cuesta -20** (reclamaciones de huéspedes). Copia la clase
`EntornoHotel` con otro nombre, cambia esa penalización, reentrena con `entrenar` (tendrás que pasar el
nuevo entorno: añade un parámetro `clase_entorno`) y comprueba que la política **sigue evitando** las obras.
Después responde: ¿qué pasaría si la penalización fuera **+5** (una "recompensa" por pasar)? Es un ejemplo
de cómo un mal diseño de la **recompensa** produce comportamientos no deseados.

## Preguntas de reflexión
1. ¿Qué papel juega `gamma` en el aprendizaje? ¿Por qué con `gamma = 0` no encuentra la meta?
2. ¿Qué ocurre si `epsilon` no baja nunca? ¿Cómo se usaría en producción un agente ya entrenado?
3. Una tabla Q funciona con 20 estados. ¿Y con millones de estados (precios, ocupación, competencia,
   fecha...)? *(pista: redes neuronales → "Deep Q-learning", Sprint 10)*.
4. Un agente que ajusta **precios dinámicos** maximizando ingresos: ¿qué recompensa mal diseñada podría
   perjudicar a los clientes o a la reputación del hotel?

## Qué has aprendido
- Agente, entorno, estado, acción, recompensa y política.
- La regla de Q-learning y los parámetros `alpha` y `gamma`.
- Explorar frente a explotar (epsilon-greedy).
- El diseño de la **recompensa** es la parte delicada: el agente hace lo que se le premia, no lo que queremos.